# T08 - Pseudo Random Number Generator (PRNG)

## Objective
Design and implement a pseudo-random number generator (PRNG) on the **CMOD A7 FPGA board** using Verilog. The generated number shall be displayed on the 4-digit 7-segment LED display, with features for reset, seeding, and periodic updates.

---

## Key Features
1. **Reset Functionality**  
   - Press **BTN0** to restart the system and generate a new seed.
   
2. **7-Segment Display**  
   - Display the pseudo-random number in decimal (e.g., `3271`) or hexadecimal (e.g., `3ab1`).  
   - First number after reset must be validated using an external seed source:  
     - Ambient temperature  
     - Ambient light intensity  
     - Thermal noise from resistor  
     - Real-time clock  
     - *Additional remarks awarded for creative seeding approaches*  

3. **Update Frequency**  
   - Refresh the displayed number at a configurable interval (e.g., 1 second).  

---

## Instructions

### FPGA Project Setup
1. **Vivado Project**  
   - Create a new Vivado project targeting the **CMOD A7-35T** FPGA.  

2. **Verilog Implementation**  
   - Design the PRNG core using a linear feedback shift register (LFSR) or other algorithm.  

3. **Testbench (Optional)**  
   - Verify PRNG output distribution (e.g., Gaussian) using statistical tests.  

4. **Constraint File (.xdc)**  
   - Map I/O pins for buttons, clock, and 7-segment display.  

5. **Bitstream Generation & Programming**  
   - Synthesize the design and program the FPGA.  

6. **Testing & Documentation**  
   - Debug using onboard LEDs and document challenges, state diagrams, and design choices.  

---

## Submission Requirements
1. **Jupyter Notebook (`T08_PRNG.ipynb`)**  
   - Include:  
     - **Verilog Code** (modules, testbenches, waveforms).  
     - **Demo Video** (≤1 minute, MP4 format).  
     - **Report** with design rationale, challenges, and lessons learned.  

2. **Additional Files**  
   - Submit `.ipynb` and video to edimension.  

---

## Challenges
1. **Application Project**  
   - Implement a real-world application (e.g., lottery system, game).  

2. **Algorithmic Project**  
   - Develop a novel PRNG variant (e.g., combining LFSR with chaos theory).  

3. **Theoretical Project**  
   - Analyze PRNG properties (e.g., entropy, periodicity).  

4. **True Random Number Generator**

*Contact the faculty and researchers for customized project ideas.*

---

## Deadline & Grading
- **Deadline**: Set in the edimension submission folder.  
- **Grading Criteria**:  
  - Innovativeness.
  - Educational values.  
  - *Bonus points for creative implementations.*  
- The report should present how AI is used in the design (if applicable).
---



---

# Start your handson's report from this line

(Complete the report, save this ipynb, and submit in edimention)

---

## Group No.:

Member name:
1. Casana Francine Flores
2. Ho I Chia  
3. Joel Yeo Yu Wei   
4. Koh Zhao En  
5. Yi Fei Yang
6. Renzo Ricardo Chua King


In [7]:
# Environment Check-Up
! vivado -version
%load_ext vivado_magic

vivado v2025.1 (64-bit)
Tool Version Limit: 2025.05
SW Build 6140274 on Wed May 21 22:58:25 MDT 2025
IP Build 6138677 on Thu May 22 03:10:11 MDT 2025
SharedData Build 6139179 on Tue May 20 17:58:58 MDT 2025
Copyright 1986-2022 Xilinx, Inc. All Rights Reserved.
Copyright 2022-2025 Advanced Micro Devices, Inc. All Rights Reserved.
The vivado_magic extension is already loaded. To reload it, use:
  %reload_ext vivado_magic


In [8]:
device_part = "xc7a35tcpg236-1"
project_name = "PRNG"
design_top = "prng"
sim_top = "prng_tb"
%vivado prj_clean

[INFO]: Removing build directory: /home/user/ug2608/Dsl_notebook/1007940 - Francine/Week 8/build_PRNG


# 2nd order MRG PRNG Module

In [10]:
%%vivado_design mrg_2nd_order
module mrg_2nd_order (
    input  wire        clk,
    input  wire        rst_n,
    input  wire        en,
    output wire [5:0]  HEX,  // Anode control
    output wire [6:0]  SEG,   // Cathode control
    output wire [31:0] next_val
);
    // Internal Registers
    reg [31:0] x_n1;          // x_{n-1}
    reg [31:0] x_n2;          // x_{n-2}
    reg [31:0] range_limited; // Internal value (0-9)
    reg [6:0]  segment_r;     // Decoder output
    // Constants
    localparam [31:0] M  = 32'hFFFFFFFF; 
    localparam [31:0] A1 = 32'd16807352;    
    localparam [31:0] A2 = 32'd48271452;

    // IO Assignments
    assign SEG = ~segment_r;  // Active Low conversion
    assign HEX = ~6'b000001;  // Driving the first digit only
 
    // PRNG Logic Signals
    wire [62:0] sum;
    wire [31:0] mod_sum;

 
    // 1. Calculate the linear combination (62-bit wide result)
    assign sum = (A1 * x_n1) + (A2 * x_n2);
    // 2. Mersenne Prime Modulo: (A + B) mod (2^k - 1)
    assign mod_sum = sum[31:0] + sum[61:32];
    assign next_val = (mod_sum > M) ? (mod_sum - M) : mod_sum;
 
    // Sequential Block: State Updates
    always @(posedge clk or negedge rst_n) begin
        if (!rst_n) begin
            x_n1          <= 32'd12345463; 
            x_n2          <= 32'd67890593;
            range_limited <= 32'd0;
        end else if (en) begin
            x_n2          <= x_n1;
            x_n1          <= next_val;
            range_limited <= next_val % 32'd10; // Result is 0-9
        end
    end
 
    // Combinational Block: 7-Segment Decoder
    // This ensures the display updates immediately when range_limited changes
    always @(*) begin
        case(range_limited)
            32'd0:   segment_r = 7'b1000000; // "0"
            32'd1:   segment_r = 7'b1111001; // "1"
            32'd2:   segment_r = 7'b0100100; // "2"
            32'd3:   segment_r = 7'b0110000; // "3"
            32'd4:   segment_r = 7'b0011001; // "4"
            32'd5:   segment_r = 7'b0010010; // "5"
            32'd6:   segment_r = 7'b0000010; // "6"
            32'd7:   segment_r = 7'b1111000; // "7"
            32'd8:   segment_r = 7'b0000000; // "8"
            32'd9:   segment_r = 7'b0010000; // "9"
            default: segment_r = 7'b1111111; // Off
        endcase
    end
 
endmodule

[WARN]: Overwriting existing design file: /home/user/ug2608/Dsl_notebook/1007940 - Francine/Week 8/build_PRNG/src/hdl/mrg_2nd_order.sv
[INFO]: Running Vivado xvlog lint:
         xvlog -sv /home/user/ug2608/Dsl_notebook/1007940 - Francine/Week 8/build_PRNG/src/hdl/mrg_2nd_order.sv
INFO: [VRFC 10-2263] Analyzing SystemVerilog file "/home/user/ug2608/Dsl_notebook/1007940 - Francine/Week 8/build_PRNG/src/hdl/mrg_2nd_order.sv" into library work
INFO: [VRFC 10-311] analyzing module mrg_2nd_order

[INFO]: Vivado xvlog lint completed with no errors (exit code 0).
[INFO]: Vivado project '/home/user/ug2608/Dsl_notebook/1007940 - Francine/Week 8/build_PRNG/PRNG/PRNG.xpr' not found; skipping auto-add.
[INFO]: The file is saved under build/src and will be picked up when you run '%vivado prj_create'.


# Testbench to verify PRNG MRG

In [5]:
%%vivado_sim mrg_2nd_order_tb
`timescale 1ns / 1ps

module mrg_2nd_order_tb();

    // Inputs
    reg clk;
    reg rst_n;
    reg en;

    // Outputs
    wire [5:0] HEX;
    wire [6:0] SEG;
    wire [31:0] next_val;

    // Instantiate the Unit Under Test (UUT)
    mrg_2nd_order uut (
        .clk(clk),
        .rst_n(rst_n),
        .en(en),
        .HEX(HEX),
        .SEG(SEG),
        .next_val(next_val)
    );

    // Clock generation (100MHz -> 10ns period)
    always #5 clk = ~clk;

    initial begin
        // Initialize Inputs
        clk = 0;
        rst_n = 0;
        en = 0;

        // Wait 20 ns for global reset to settle
        #20;
        
        // Release reset
        rst_n = 1;
        #10;
        
        // Enable the PRNG for 20 cycles
        en = 1;
        #200;
        
        // Disable and wait
        en = 0;
        #50;
        
        // Enable again to see it resume
        en = 1;
        #100;

        // Finish simulation
        $display("Simulation Finished.");
        $finish;
    end

    // Monitor the internal values to see the sequence
    // Use 'uut.' to peek inside the module
    initial begin
        $display("Time\t En \t Val(0-9) \t 7-Seg(Hex)");
        $monitor("%0t\t %b \t %d \t\t %h", $time, en, uut.range_limited, SEG);
    end

endmodule

[INFO]: Creating testbench file: /home/user/ug2608/Dsl_notebook/1007940 - Francine/Week 8/build_PRNG/src/tb/mrg_2nd_order_tb.sv
[INFO]: Running Vivado xvlog lint:
         xvlog -sv /home/user/ug2608/Dsl_notebook/1007940 - Francine/Week 8/build_PRNG/src/hdl/prng.sv /home/user/ug2608/Dsl_notebook/1007940 - Francine/Week 8/build_PRNG/src/hdl/mrg_2nd_order.sv /home/user/ug2608/Dsl_notebook/1007940 - Francine/Week 8/build_PRNG/src/tb/mrg_2nd_order_tb.sv /home/user/ug2608/Dsl_notebook/1007940 - Francine/Week 8/build_PRNG/src/tb/logicf_tb.sv
INFO: [VRFC 10-2263] Analyzing SystemVerilog file "/home/user/ug2608/Dsl_notebook/1007940 - Francine/Week 8/build_PRNG/src/hdl/prng.sv" into library work
INFO: [VRFC 10-311] analyzing module prng_dual_lfsr
INFO: [VRFC 10-2263] Analyzing SystemVerilog file "/home/user/ug2608/Dsl_notebook/1007940 - Francine/Week 8/build_PRNG/src/hdl/mrg_2nd_order.sv" into library work
INFO: [VRFC 10-311] analyzing module mrg_2nd_order
INFO: [VRFC 10-2263] Analyzing SystemV

# Sensor-Driven ADC Seeding with UART Transmission of PRNG Output
This implementation enhances PRNG randomness by seeding it with real-world entropy obtained from light and temperature sensors via an MCP3202 ADC. The sampled analogue data introduces additional unpredictability into the seed. Although not integrated into the final hardware, this approach was validated on a separate development board, with generated values transmitted through UART for observation. 

# Top Module
The top module integrates ADC-based seeding, PRNG generation, and output display. Light and temperature sensor data are sampled via the MCP3202 ADC to initialise the PRNG seed, after which random values are continuously generated. Board interaction is enabled through btn[0], which performs an active-low system reset, while sw[0] selects the value shown on the 7-segment display (seed1 from light sensor or seed2 from temperature sensor). The system outputs data via both the 7-segment display and UART, allowing real-time observation of sensor readings and generated random numbers.

In [ ]:
%%vivado_design top_module
module top_module(
    input           sysclk,
    input   [1:0]   btn,
    input   [1:0]   sw,
    output  [7:0]   seg,
    output  [3:0]   hex,
    output          uart_rxd_out,
    output          adc_din,
    output          adc_csn,
    output          adc_clk,
    input           adc_dout    
);

//================ RESET =================
wire rstn = ~btn[0];

//================ CLOCKS =================
wire clk_adc_tri, clk_adc_spi, clk_uart_clk, clk_seg_clk;

//================ ADC =================
wire [11:0] adc_data;
wire        adc_valid;
reg mode_sel;
wire [1:0] adc_mode = {1'b0, mode_sel};

always @(posedge clk_adc_spi or negedge rstn) begin
    if (!rstn)
        mode_sel <= 0;
    else if (adc_valid)
        mode_sel <= ~mode_sel;
end

reg adc_tri_d;
always @(posedge clk_adc_spi) adc_tri_d <= clk_adc_tri;
wire adc_ready = clk_adc_tri & ~adc_tri_d;

//================ ADC SYNC =================
reg [11:0] adc_sync1, adc_sync2;
always @(posedge clk_uart_clk) begin
    adc_sync1 <= adc_data;
    adc_sync2 <= adc_sync1;
end
wire [11:0] adc_data_uart = adc_sync2;

//sync adc_valid into uart clock domain
reg adc_valid_s1, adc_valid_s2, adc_valid_s3;
always @(posedge clk_uart_clk or negedge rstn) begin
    if (!rstn) begin
        adc_valid_s1 <= 0;
        adc_valid_s2 <= 0;
        adc_valid_s3 <= 0;
    end else begin
        adc_valid_s1 <= adc_valid;
        adc_valid_s2 <= adc_valid_s1;
        adc_valid_s3 <= adc_valid_s2;
    end
end
// rising edge detect of synced valid pulse
wire adc_valid_uart = adc_valid_s2 & ~adc_valid_s3;

//================ capture mode before toggling =================
reg mode_at_capture;
always @(posedge clk_adc_spi or negedge rstn) begin
    if (!rstn)
        mode_at_capture <= 0;
    else if (adc_valid)
        mode_at_capture <= mode_sel;  // latch BEFORE toggle
end

reg mac_sync1, mac_sync2;
always @(posedge clk_uart_clk) begin
    mac_sync1 <= mode_at_capture;
    mac_sync2 <= mac_sync1;
end
wire mode_uart = mac_sync2;

//================ ADC DEMUX =================
reg [11:0] adc_light;
reg [11:0] adc_temp;

always @(posedge clk_uart_clk) begin
    if (adc_valid_uart) begin        // use synced valid, not raw
        if (mode_uart == 1'b0)
            adc_light <= adc_data_uart;
        else
            adc_temp  <= adc_data_uart;
    end
end

//================ SEED =================
reg [30:0] seed1_reg, seed2_reg;
reg seeded;
reg light_valid, temp_valid;

always @(posedge clk_uart_clk or negedge rstn) begin
    if (!rstn) begin
        seed1_reg   <= 0;
        seed2_reg   <= 0;
        seeded      <= 0;
        light_valid <= 0;
        temp_valid  <= 0;
    end 
    else if (adc_valid_uart && !seeded) begin   // <-- adc_valid_uart here
        if (mode_uart == 0)
            light_valid <= 1;
        else
            temp_valid <= 1;

        if ((light_valid || (mode_uart == 0)) &&
            (temp_valid  || (mode_uart == 1))) begin
            seed1_reg <= {19'b0, adc_light};
            seed2_reg <= {19'b0, adc_temp};
            seeded    <= 1;
        end
    end
end

//================ SEED VALID PULSE =================
reg seeded_d;
always @(posedge clk_uart_clk or negedge rstn) begin
    if (!rstn)
        seeded_d <= 0;
    else
        seeded_d <= seeded;
end
wire seed_valid = seeded & ~seeded_d;

//================ PRNG =================
wire [30:0] prng_out;

mrg prng_u0 (
    .clk       (clk_uart_clk),
    .rst_n     (rstn),
    .en        (1'b1),
    .seed1     (seed1_reg),
    .seed2     (seed2_reg),
    .seed_valid(seed_valid),
    .prng_out  (prng_out)
);

//================ DISPLAY =================
reg btn_d;
always @(posedge clk_uart_clk) btn_d <= btn[1];
wire btn_pressed = btn[1] & ~btn_d;
reg [15:0] display_value;

always @(posedge sysclk or negedge rstn) begin
    if (!rstn)
        display_value <= 0;
    else begin
        if (sw[0])
            display_value <= seed1_reg[15:0];
        else
            display_value <= seed2_reg[15:0];
    end
end

//================ HEX TO ASCII =================
function [7:0] hex_to_ascii;
    input [3:0] nibble;
    begin
        if (nibble < 10)
            hex_to_ascii = nibble + "0";
        else
            hex_to_ascii = nibble - 10 + "A";
    end
endfunction

//================ UART ADC STREAM =================
// FIX: widened to [5:0] to hold 38 states
reg [5:0]  state;
reg [11:0] light_value;
reg [11:0] temp_value;
reg [30:0] rng_value;

reg        uart_ready;
wire       uart_valid;
reg [7:0]  uart_data;
reg        started;

// Format: "TEMP:XXXX LIGHT:XXXX PRNG:XXXXXXXX\r\n\r\n"
localparam S_IDLE  = 0,
           S_T1    = 1,   // 'T'
           S_T2    = 2,   // 'E'
           S_T3    = 3,   // 'M'
           S_TC    = 4,   // 'P'
           S_TD3   = 5,   // ':'
           S_TD2   = 6,
           S_TD1   = 7,
           S_TD0   = 8,
           S_SP1   = 9,   // ' '
           S_L0    = 10,  // 'L'
           S_L1    = 11,  // 'I'
           S_L2    = 12,  // 'G'
           S_L3    = 13,  // 'H'
           S_L4    = 14,  // 'T'
           S_LC    = 15,  // ':'
           S_LD3   = 16,
           S_LD2   = 17,
           S_LD1   = 18,
           S_LD0   = 19,
           S_SP2   = 20,  // ' '
           S_P0    = 21,  // 'P'
           S_P1    = 22,  // 'R'
           S_P2    = 23,  // 'N'
           S_P3    = 24,  // 'G'
           S_PC    = 25,  // ':'
           S_RNG7  = 26,
           S_RNG6  = 27,
           S_RNG5  = 28,
           S_RNG4  = 29,
           S_RNG3  = 30,
           S_RNG2  = 31,
           S_RNG1  = 32,
           S_RNG0  = 33,
           S_CR    = 34,  // \r
           S_LF    = 35,  // \n
           S_CR2   = 36,  // \r  (blank line)
           S_LF2   = 37;  // \n

always @(posedge clk_uart_clk or negedge rstn) begin
    if (!rstn) begin
        state      <= S_IDLE;
        uart_ready <= 0;
        uart_data  <= 0;
        started    <= 0;
    end else begin
        uart_ready <= 0;

        if (!started) begin
            temp_value  <= adc_temp;
            light_value <= adc_light;
            rng_value   <= prng_out;
            uart_data   <= "T";
            uart_ready  <= 1;
            state       <= S_T2;
            started     <= 1;
        end

        else if (uart_valid) begin
            case (state)

                S_IDLE: begin
                    temp_value  <= adc_temp;
                    light_value <= adc_light;
                    rng_value   <= prng_out;
                    uart_data   <= "T";
                    uart_ready  <= 1;
                    state       <= S_T2;
                end

                // --- TEMP: ---
                S_T2:  begin uart_data <= "E"; uart_ready <= 1; state <= S_T3;  end
                S_T3:  begin uart_data <= "M"; uart_ready <= 1; state <= S_TC;  end
                S_TC:  begin uart_data <= "P"; uart_ready <= 1; state <= S_TD3; end
                S_TD3: begin uart_data <= ":"; uart_ready <= 1; state <= S_TD2; end

                S_TD2: begin uart_data <= (temp_value / 1000) + "0";          uart_ready <= 1; state <= S_TD1; end
                S_TD1: begin uart_data <= ((temp_value % 1000) / 100) + "0";  uart_ready <= 1; state <= S_TD0; end
                S_TD0: begin uart_data <= ((temp_value % 100) / 10) + "0";    uart_ready <= 1; state <= S_SP1; end
                S_SP1: begin uart_data <= (temp_value % 10) + "0";            uart_ready <= 1; state <= S_L0;  end

                // --- LIGHT: ---
                S_L0:  begin uart_data <= " ";  uart_ready <= 1; state <= S_L1;  end
                S_L1:  begin uart_data <= "L";  uart_ready <= 1; state <= S_L2;  end
                S_L2:  begin uart_data <= "I";  uart_ready <= 1; state <= S_L3;  end
                S_L3:  begin uart_data <= "G";  uart_ready <= 1; state <= S_L4;  end
                S_L4:  begin uart_data <= "H";  uart_ready <= 1; state <= S_LC;  end
                S_LC:  begin uart_data <= "T";  uart_ready <= 1; state <= S_LD3; end
                S_LD3: begin uart_data <= ":";  uart_ready <= 1; state <= S_LD2; end

                S_LD2: begin uart_data <= (light_value / 1000) + "0";         uart_ready <= 1; state <= S_LD1; end
                S_LD1: begin uart_data <= ((light_value % 1000) / 100) + "0"; uart_ready <= 1; state <= S_LD0; end
                S_LD0: begin uart_data <= ((light_value % 100) / 10) + "0";   uart_ready <= 1; state <= S_SP2; end
                S_SP2: begin uart_data <= (light_value % 10) + "0";           uart_ready <= 1; state <= S_P0;  end

                // --- PRNG: ---
                S_P0:  begin uart_data <= " ";  uart_ready <= 1; state <= S_P1;   end
                S_P1:  begin uart_data <= "P";  uart_ready <= 1; state <= S_P2;   end
                S_P2:  begin uart_data <= "R";  uart_ready <= 1; state <= S_P3;   end
                S_P3:  begin uart_data <= "N";  uart_ready <= 1; state <= S_PC;   end
                S_PC:  begin uart_data <= "G";  uart_ready <= 1; state <= S_RNG7; end
                S_RNG7: begin uart_data <= ":"; uart_ready <= 1; state <= S_RNG6; end

                S_RNG6: begin uart_data <= hex_to_ascii(rng_value[30:27]); uart_ready <= 1; state <= S_RNG5; end
                S_RNG5: begin uart_data <= hex_to_ascii(rng_value[26:23]); uart_ready <= 1; state <= S_RNG4; end
                S_RNG4: begin uart_data <= hex_to_ascii(rng_value[22:19]); uart_ready <= 1; state <= S_RNG3; end
                S_RNG3: begin uart_data <= hex_to_ascii(rng_value[18:15]); uart_ready <= 1; state <= S_RNG2; end
                S_RNG2: begin uart_data <= hex_to_ascii(rng_value[14:11]); uart_ready <= 1; state <= S_RNG1; end
                S_RNG1: begin uart_data <= hex_to_ascii(rng_value[10:7]);  uart_ready <= 1; state <= S_RNG0; end
                S_RNG0: begin uart_data <= hex_to_ascii(rng_value[6:3]);   uart_ready <= 1; state <= S_CR;   end
                S_CR:   begin uart_data <= hex_to_ascii(rng_value[2:0]);   uart_ready <= 1; state <= S_LF;   end

                S_LF:  begin uart_data <= 8'h0D; uart_ready <= 1; state <= S_CR2; end
                S_CR2: begin uart_data <= 8'h0A; uart_ready <= 1; state <= S_LF2; end
                S_LF2: begin uart_data <= 8'h0D; uart_ready <= 1; state <= S_IDLE; end  // blank line \r\n

                default: state <= S_IDLE;
            endcase
        end
    end
end

//================ UART =================
drv_uart_tx uart_u0(
    .clk     (clk_uart_clk),
    .ap_rstn (rstn),
    .ap_ready(uart_ready),
    .ap_valid(uart_valid),
    .tx      (uart_rxd_out),
    .parity  (1'b0),
    .data    (uart_data)
);

//================ 7-SEG =================
drv_7segment seg_u0(
    .i_rstn(rstn), 
    .i_clk (clk_seg_clk),
    .i_hex3(display_value[15:12]), 
    .i_hex2(display_value[11:8]), 
    .i_hex1(display_value[7:4]), 
    .i_hex0(display_value[3:0]),
    .o_seg (seg),
    .o_hex (hex)
);

//================ CLOCK DIV =================
clk_div #( .CLK_IN_HZ(12_000_000), .CLK_OUT_HZ(1)         ) div0 (sysclk, rstn, clk_adc_tri);
clk_div #( .CLK_IN_HZ(12_000_000), .CLK_OUT_HZ(1_000_000) ) div1 (sysclk, rstn, clk_adc_spi);
clk_div #( .CLK_IN_HZ(12_000_000), .CLK_OUT_HZ(9600)      ) div3 (sysclk, rstn, clk_uart_clk);
clk_div #( .CLK_IN_HZ(12_000_000), .CLK_OUT_HZ(1000)      ) div4 (sysclk, rstn, clk_seg_clk);

//================ ADC =================
drv_mcp3202 adc_u0 (
    .rstn    (rstn),
    .clk     (clk_adc_spi),
    .ap_ready(adc_ready),
    .ap_valid(adc_valid),
    .mode    (adc_mode),
    .data    (adc_data),
    .adc_din (adc_din),
    .adc_dout(adc_dout),
    .adc_clk (adc_clk),
    .adc_cs  (adc_csn)
);

endmodule

# PRNG MRG Module

In [ ]:
%%vivado_design mrg
module mrg(
    input  wire        clk,
    input  wire        rst_n,
    input  wire        en,

    input  wire [30:0] seed1,
    input  wire [30:0] seed2,
    input  wire        seed_valid,

    output wire [30:0] prng_out
);

    reg [30:0] x_n1;
    reg [30:0] x_n2;
    reg [30:0] prng_reg;

    localparam [30:0] M  = 31'h7FFFFFFF; 
    localparam [30:0] A1 = 31'd16807836;    
    localparam [30:0] A2 = 31'd48271531;

    wire [61:0] sum;
    wire [30:0] mod_sum1;
    wire [30:0] mod_sum2;
    wire [30:0] next_val;

    // Multiply and accumulate
    assign sum = (A1 * x_n1) + (A2 * x_n2);

    // First reduction
    assign mod_sum1 = sum[30:0] + sum[61:31];

    // Second reduction (fixes overflow edge case)
    assign mod_sum2 = mod_sum1 + (mod_sum1 >> 31);

    // Final modulo correction
    assign next_val = (mod_sum2 > M) ? (mod_sum2 - M) : mod_sum2;

    // Output register (synchronised)
    assign prng_out = prng_reg;

    always @(posedge clk or negedge rst_n) begin
        if (!rst_n) begin
            x_n1     <= 31'd1;
            x_n2     <= 31'd2;
            prng_reg <= 31'd0;
        end 
        else if (seed_valid) begin
            x_n1     <= (seed1 == 0) ? 31'd1 : seed1;
            x_n2     <= (seed2 == 0) ? 31'd2 : seed2;
            prng_reg <= 31'd0;
        end
        else if (en) begin
            x_n2     <= x_n1;
            x_n1     <= next_val;
            prng_reg <= next_val;
        end
    end

endmodule

# drv_uart_tx module: 
UART communication between CMOD A7 and computer (9600 baudrate).
Protocol sequence: IDLE -> START -> TRANSFER DATA -> PARITY -> STOP -> IDLE

In [ ]:
%%vivado_design drv_uart_tx
module drv_uart_tx(
    input   clk,
    input   ap_rstn,
    input   ap_ready,
    output  reg ap_valid,
    output  reg tx,
    input   parity,
    input  [7:0] data
);

localparam  FSM_IDLE = 3'b000,
            FSM_STAR = 3'b001,
            FSM_TRSF = 3'b010,
            FSM_PARI = 3'b011,
            FSM_STOP = 3'b100;

reg [2:0] fsm_statu;
reg [2:0] fsm_next;
reg [2:0] cnter;

// STATE
always @(posedge clk or negedge ap_rstn) begin
    if (!ap_rstn)
        fsm_statu <= FSM_IDLE;
    else
        fsm_statu <= fsm_next;
end

// NEXT STATE
always @(*) begin
    case (fsm_statu)
        FSM_IDLE: fsm_next = (ap_ready) ? FSM_STAR : FSM_IDLE;
        FSM_STAR: fsm_next = FSM_TRSF;
        FSM_TRSF: fsm_next = (cnter == 3'h7) ?
                             (parity ? FSM_PARI : FSM_STOP) : FSM_TRSF;
        FSM_PARI: fsm_next = FSM_STOP;
        FSM_STOP: fsm_next = FSM_IDLE;   // ? FIXED
        default:  fsm_next = FSM_IDLE;
    endcase
end

// OUTPUT
always @(posedge clk or negedge ap_rstn) begin
    if (!ap_rstn) begin
        ap_valid <= 1'b0;
        tx       <= 1'b1;
        cnter    <= 3'h0;
    end else begin
        case (fsm_statu)

            FSM_IDLE: begin
                tx <= 1'b1;
                ap_valid <= 1'b0;
            end

            FSM_STAR: begin
                tx <= 1'b0;
                cnter <= 3'h0;
            end

            FSM_TRSF: begin
                tx <= data[cnter];
                cnter <= cnter + 1'b1;
            end

            FSM_PARI: tx <= (^data);

            FSM_STOP: begin
                tx <= 1'b1;
                ap_valid <= 1'b1;
                cnter <= 3'h0;   
            end

        endcase
    end
end

endmodule

# drv_7segment Module
Displays the selected seed value on the 7-segment display.

In [ ]:
%%vivado_design drv_7segment
module drv_7segment(
    input           i_rstn, //  ASync Reset;
    input           i_clk,  //1Khz - Drive Hex Scan;
    input  [3:0]    i_hex3, // HEX3 - Data;
    input  [3:0]    i_hex2, // HEX2 - Data;
    input  [3:0]    i_hex1, // HEX1 - Data;
    input  [3:0]    i_hex0, // HEX0 - Data;
    output [7:0]    o_seg,  // Output Segment Data;
    output [3:0]    o_hex   // Output Hex (Digits) Data;
);

reg [1:0]   r_cnter     ;   //counter for which 7-segment digit is active 2-bits = max 4 digits
reg [7:0]   r_seg       ;   //7-segment value to display from 0-F (hex)
reg [3:0]   r_hex       ;   //shows the current active 7-segment digit
reg [3:0]   r_hexdata   ;   //the hex value for current active digit (each digit displays only 4-bits of r_adc_data)

assign o_hex = ~ r_hex;     //digit select is active-LOW for CMOD A7, thus requires inverting
assign o_seg = r_seg;       //passes r_seg data to output o_seg

always @(negedge i_rstn, posedge i_clk) begin   //runs every 1kHz (when i_clk changes from LOW to HIGH) or when btn[0] is pressed
    if (!i_rstn)    r_cnter <= 2'b00;           //reset 
    else            r_cnter <= r_cnter + 1'b1;  //counter for r_cnter to cycle through the 4 digits
end

always @(r_cnter) begin                 //runs every 1kHz (when i_clk changes from LOW to HIGH) or when btn[0] is pressed
    case (r_cnter)
        2'b00   : r_hex <= 4'b0001;     //last 7-segment digit is selected
        2'b01   : r_hex <= 4'b0010;     //3rd 7-segment digit is selected
        2'b10   : r_hex <= 4'b0100;     //2nd 7-segment digit is selected
        2'b11   : r_hex <= 4'b1000;     //1st 7-segment digit is selected
        default : r_hex <= 4'b0000;     //all digits are off
    endcase
end

always @(r_cnter) begin                     //runs every 1kHz (when i_clk changes from LOW to HIGH) or when btn[0] is pressed
    case (r_cnter)
        2'b00   : r_hexdata <= i_hex0;      //passes the hex0 data
        2'b01   : r_hexdata <= i_hex1;      //passes the hex1 data
        2'b10   : r_hexdata <= i_hex2;      //passes the hex2 data
        2'b11   : r_hexdata <= i_hex3;      //passes the hex3 data
        default : r_hexdata <= 4'h0;
    endcase
end 

always @(*) begin
    case (r_hexdata)
        4'h0: r_seg = 8'b0_0111111; // 0
        4'h1: r_seg = 8'b0_0000110; // 1
        4'h2: r_seg = 8'b0_1011011; // 2
        4'h3: r_seg = 8'b0_1001111; // 3
        4'h4: r_seg = 8'b0_1100110; // 4
        4'h5: r_seg = 8'b0_1101101; // 5
        4'h6: r_seg = 8'b0_1111101; // 6
        4'h7: r_seg = 8'b0_0000111; // 7
        4'h8: r_seg = 8'b0_1111111; // 8
        4'h9: r_seg = 8'b0_1101111; // 9
        4'hA: r_seg = 8'b0_1110111; // A
        4'hB: r_seg = 8'b0_1111100; // b
        4'hC: r_seg = 8'b0_0111001; // C
        4'hD: r_seg = 8'b0_1011110; // d
        4'hE: r_seg = 8'b0_1111001; // E
        4'hF: r_seg = 8'b0_1110001; // F
        default: r_seg = 8'b0_0000000; // all off
    endcase
end

endmodule


# Clock Divier Module

In [ ]:
// ============================================================================
// Clock Divider
// Generates a lower-frequency clock o_clkout from input clock i_clkin.
// The output frequency is configured by CLK_IN_HZ and CLK_OUT_HZ.
// ============================================================================
%%vivado_design clk_div
module clk_div #(
    // ===== User-configurable parameters (in Hz) =============================
    parameter integer CLK_IN_HZ  = 12_000_000, // Input clock frequency
    parameter integer CLK_OUT_HZ = 1_000       // Desired output clock frequency
)(
    input  wire i_clkin,  // Source clock input
    input  wire i_rstn,   // Asynchronous active-low reset
    output wire o_clkout  // Divided clock output
);

    // ===== Derived parameters (internal use only) ==========================
    localparam integer HALF_PERIOD_COUNT = CLK_IN_HZ / (2 * CLK_OUT_HZ);
    localparam integer CNT_WIDTH         = $clog2(HALF_PERIOD_COUNT);

    reg [CNT_WIDTH-1:0] cnt;      // Counter for clock division
    reg                 clkout_r; // Registered divided clock

    assign o_clkout = clkout_r;

    // Asynchronous active-low reset, synchronous clock divider logic
    always @(posedge i_clkin or negedge i_rstn) begin
        if (!i_rstn) begin
            // Reset counter and output clock
            cnt      <= 0;
            clkout_r <= 1'b0;
        end
        else begin
            if (cnt == HALF_PERIOD_COUNT - 1) begin
                // Toggle output clock and reset counter at terminal count
                cnt      <= 0;
                clkout_r <= ~clkout_r;
            end
            else begin
                // Increment counter
                cnt <= cnt + 1'b1;
            end
        end
    end

endmodule



# drv_mcp3202:
Interfaces with the MCP3202 ADC to acquire sensor data for PRNG seeding.

In [ ]:
%%vivado_design drv_mcp3202
module drv_mcp3202(
    input        rstn,
    input        clk,           //clk_adc_spi from top_module (1MHz)
    input        ap_ready,
    output reg   ap_valid,
    input  [1:0] mode,
    output [11:0] data,

    // === SPI pins to MCP3202 ===
    // adc_din  : FPGA -> ADC (MOSI, goes to MCP3202 DIN)
    // adc_dout : ADC  -> FPGA (MISO, comes from MCP3202 DOUT)
    output reg  adc_din,
    input       adc_dout,
    output      adc_clk,    //SPI clk to ADC
    output reg  adc_cs      //SPI chip select
);

    // 4-bit control word to send to MCP3202
    wire [3:0]  Data_Transmit;  // [START, SGL/DIFF, ODD/SIGN, MSBF]
    // 1 dummy bit + 12 data bits from MCP3202
    reg  [12:0] Data_Receive;

    assign Data_Transmit[3]   = 1'b1;      // START bit
    assign Data_Transmit[0]   = 1'b1;      // MSBF (always 1 for MCP3202)
    assign Data_Transmit[2:1] = mode;      // channel / mode select

    //FSM states
    reg [1:0] fsm_statu, fsm_next;
    localparam FSM_IDLE = 2'b00;
    localparam FSM_WRIT = 2'b10;
    localparam FSM_READ = 2'b11;
    localparam FSM_STOP = 2'b01;

    reg [1:0] cnter_writ;   //adc_din bits counter
    reg [3:0] cnter_read;   //tracks which adc_dout bit to receive

    //===========================================================
    // FSM state register
    //===========================================================
    always @(posedge clk or negedge rstn) begin     //runs at 1MHz (when clk changes from LOW to HIGH) or when btn[0] is pressed
        if (!rstn)                                  //if btn[0] is pressed, make FSM idle
            fsm_statu <= FSM_IDLE;
        else                                        //if btn[0] is not pressed, make FSM go to next state
            fsm_statu <= fsm_next;
    end

    //===========================================================
    // FSM next-state logic
    //===========================================================
    always @(*) begin
        if (!rstn) begin            //if btn[0] is pressed, make FSM idle
            fsm_next = FSM_IDLE;
        end else begin              //if btn[0] is not pressed, check what state FSM is in currently, then sequentially set the next state based on SPI protocol (IDLE -> WRITE (send 4 control bits) -> READ (receive 13 bits) -> STOP -> IDLE)
            case (fsm_statu)
                FSM_IDLE: fsm_next = (ap_ready)          ? FSM_WRIT : FSM_IDLE;
                FSM_WRIT: fsm_next = (cnter_writ == 2'd0)? FSM_READ : FSM_WRIT;
                FSM_READ: fsm_next = (cnter_read == 4'd0)? FSM_STOP : FSM_READ;
                FSM_STOP: fsm_next = (!ap_ready)         ? FSM_STOP : FSM_IDLE;
                default:  fsm_next = FSM_IDLE;
            endcase
        end
    end

    //===========================================================
    // FSM outputs - SPI write (MOSI, CS) on falling edge of clk
    //===========================================================
    always @(negedge clk or negedge rstn) begin
        if (!rstn) begin
            cnter_writ <= 2'd3;
            adc_din    <= 1'b1;
            adc_cs     <= 1'b1;
        end else begin
            case (fsm_statu)
                FSM_IDLE: begin
                    cnter_writ <= 2'd3;
                    adc_din    <= 1'b1;
                    adc_cs     <= 1'b1;
                end
                FSM_WRIT: begin
                    adc_cs     <= 1'b0;
                    adc_din    <= Data_Transmit[cnter_writ];  // MOSI
                    cnter_writ <= cnter_writ - 1'b1;
                end
                FSM_READ: begin
                    adc_cs     <= 1'b0;
                    adc_din    <= 1'b1;                       // keep MOSI high
                end
                FSM_STOP: begin
                    adc_cs     <= 1'b1;
                end
                default: ; // do nothing
            endcase
        end
    end

    //===========================================================
    // FSM outputs - SPI read (MISO) and ap_valid on rising edge
    //===========================================================
    always @(posedge clk or negedge rstn) begin
        if (!rstn) begin
            cnter_read   <= 4'd13;
            Data_Receive <= 13'h0000;
            ap_valid     <= 1'b0;
        end else begin
            case (fsm_statu)
                FSM_IDLE: begin
                    ap_valid   <= 1'b0;
                    cnter_read <= 4'd13;
                end

                FSM_WRIT: begin
                    Data_Receive <= 13'h0000;
                    ap_valid     <= 1'b0;
                end

                FSM_READ: begin
                    cnter_read                 <= cnter_read - 1'b1;
                    Data_Receive[cnter_read]   <= adc_dout;   // sample MISO
                    ap_valid                   <= 1'b0;
                end

                FSM_STOP: begin
                    ap_valid <= 1'b1;  // one conversion word ready
                end

                default: begin
                    ap_valid <= 1'b0;
                end
            endcase
        end
    end

    //===========================================================
    // SPI clock and data output
    //===========================================================
    assign adc_clk = clk | adc_cs;         // stop toggling when CS is high
    assign data    = Data_Receive[11:0];   // 12-bit ADC result

endmodule



**Constraints file**

In [ ]:
%%vivado_xdc cmod_pins
# ================= CLOCK =================
# 12 MHz Clock Signal
set_property -dict { PACKAGE_PIN L17 IOSTANDARD LVCMOS33 } [get_ports { sysclk }]
create_clock -name sys_clk -period 83.33 [get_ports { sysclk }]

# ================= BUTTONS =================
set_property -dict { PACKAGE_PIN A18 IOSTANDARD LVCMOS33 } [get_ports { btn[0] }]
set_property -dict { PACKAGE_PIN B18 IOSTANDARD LVCMOS33 } [get_ports { btn[1] }]


# ================= SWITCH ===================
set_property -dict { PACKAGE_PIN V8 IOSTANDARD LVCMOS33 } [get_ports { sw[0] }]
# ================= 7-SEGMENT =================
# Segments
set_property -dict { PACKAGE_PIN B15 IOSTANDARD LVCMOS33 } [get_ports { seg[7] }]
set_property -dict { PACKAGE_PIN K3  IOSTANDARD LVCMOS33 } [get_ports { seg[6] }]
set_property -dict { PACKAGE_PIN A14 IOSTANDARD LVCMOS33 } [get_ports { seg[5] }]
set_property -dict { PACKAGE_PIN K2  IOSTANDARD LVCMOS33 } [get_ports { seg[4] }]
set_property -dict { PACKAGE_PIN J3  IOSTANDARD LVCMOS33 } [get_ports { seg[3] }]
set_property -dict { PACKAGE_PIN H1  IOSTANDARD LVCMOS33 } [get_ports { seg[2] }]
set_property -dict { PACKAGE_PIN A16 IOSTANDARD LVCMOS33 } [get_ports { seg[1] }]
set_property -dict { PACKAGE_PIN J1  IOSTANDARD LVCMOS33 } [get_ports { seg[0] }]

# Digits
set_property -dict { PACKAGE_PIN L2  IOSTANDARD LVCMOS33 } [get_ports { hex[3] }]
set_property -dict { PACKAGE_PIN L1  IOSTANDARD LVCMOS33 } [get_ports { hex[2] }]
set_property -dict { PACKAGE_PIN A15 IOSTANDARD LVCMOS33 } [get_ports { hex[1] }]
set_property -dict { PACKAGE_PIN C15 IOSTANDARD LVCMOS33 } [get_ports { hex[0] }]

# ================= ADC (MCP3202 via PMOD JA) =================
set_property -dict { PACKAGE_PIN H17 IOSTANDARD LVCMOS33 } [get_ports { adc_din  }]
set_property -dict { PACKAGE_PIN H19 IOSTANDARD LVCMOS33 } [get_ports { adc_csn  }]
set_property -dict { PACKAGE_PIN J19 IOSTANDARD LVCMOS33 } [get_ports { adc_clk  }]
set_property -dict { PACKAGE_PIN K18 IOSTANDARD LVCMOS33 } [get_ports { adc_dout }]

# ================= UART =================
set_property -dict { PACKAGE_PIN J18 IOSTANDARD LVCMOS33 } [get_ports { uart_rxd_out }]


In [ ]:
%vivado prj_create

In [ ]:
%vivado sim_rtl

In [ ]:
%vivado syn
%vivado imp

# Testing:
Over 2 million samples were generated and stored in a file named 2MIL.txt for analysis.

**File Reading**

In [ ]:
rng_values = []

with open("2MIL.txt", "r") as f:
    next(f)  # skip first line

    prev_line = None
    for line in f:
        if prev_line is not None:
            try:
                hex_str = prev_line.strip().split("RNG:")[1]
                value = int(hex_str, 16)
                rng_values.append(value)
            except:
                pass
        prev_line = line



print(rng_values[-1])
print(len(rng_values))


**Histogram and Chi-Square**

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import chisquare

data = np.array(rng_values, dtype=np.uint32)

df = pd.DataFrame(data, columns=["PRNG"])

print(f"Loaded {len(df)} samples")

# ================== 1. HISTOGRAM ==================
plt.figure()
plt.hist(
    df["PRNG"],
    bins=NUM_BINS
)

plt.title("Raw 32-bit Distribution (Binned)")
plt.xlabel("Value")
plt.ylabel("Count")
plt.grid()
plt.show()

# ================== 2. CHI-SQUARE ==================
expected = [len(df)/NUM_BINS] * NUM_BINS
chi2, p = chisquare(counts, expected)

print("\nChi-square test:")
print(f"Chi2 = {chi2:.2f}")
print(f"p-value = {p:.4f}")



**Q-Q plot of the generated values**

In [ ]:
from scipy import stats
import matplotlib.pyplot as plt
import matplotlib as mpl
import numpy as np
from scipy.stats import norm

# ================== STYLE ==================
mpl.rcParams['font.size'] = 16
mpl.rcParams['font.weight'] = 'bold'
mpl.rcParams['axes.labelsize'] = 20
mpl.rcParams['axes.labelweight'] = 'bold'
mpl.rcParams['legend.fontsize'] = 14

# ================== STEP 1: NORMALIZE ==================
x = np.array(rng_values, dtype=np.uint32)
u = (x + 0.5) / (2**32)

# ================== VECTORISED BOX-MULLER ==================
if len(u) % 2 != 0: #ensures even length
    u = u[:-1]

u1 = u[0::2]
u2 = u[1::2]

mask = u1 > 1e-12
u1 = u1[mask]
u2 = u2[mask]

r = np.sqrt(-2 * np.log(u1))
theta = 2 * np.pi * u2

z1 = r * np.cos(theta)
z2 = r * np.sin(theta)

gaussian_numbers = np.concatenate([z1, z2])


# =============== Q-Q Plot ==============================
stats.probplot(gaussian_numbers, dist="norm", plot=plt)
plt.title("Q-Q Plot (Normality Check)")
plt.show()

**Spectral Test using 10000 samples**

In [ ]:

import numpy as np
import matplotlib.pyplot as plt

subset = rng_values[:10000]
data = np.array(subset, dtype=np.float64)

# normalize properly
data = data / (2**31 - 1)

print(f"Loaded {len(data)} samples")

# ================== 2D ==================
x2 = data[:-1]
y2 = data[1:]

plt.figure()
plt.scatter(x2, y2, s=2, alpha=0.5)
plt.title("Spectral Test (2D): (x_n, x_{n+1})")
plt.xlabel("x_n")
plt.ylabel("x_{n+1}")
plt.axis('equal')
plt.grid()
plt.show()

# ================== 3D ==================
from mpl_toolkits.mplot3d import Axes3D

x3 = data[:-2]
y3 = data[1:-1]
z3 = data[2:]

fig = plt.figure()
ax = fig.add_subplot(111, projection='3d')
ax.scatter(x3, y3, z3, s=2, alpha=0.5)

ax.set_title("Spectral Test (3D)")
ax.set_xlabel("x_n")
ax.set_ylabel("x_{n+1}")
ax.set_zlabel("x_{n+2}")
ax.set_box_aspect([1,1,1])

plt.show()

# ================== CORRELATION ==================
corr = np.corrcoef(data[:-1], data[1:])[0,1]
print(f"\nCorrelation between x_n and x_(n+1): {corr:.6f}")


**Gaussian Distribution**

In [ ]:

import matplotlib.pyplot as plt
import matplotlib as mpl
import numpy as np
from scipy.stats import norm

# ================== STYLE ==================
mpl.rcParams['font.size'] = 16
mpl.rcParams['font.weight'] = 'bold'
mpl.rcParams['axes.labelsize'] = 20
mpl.rcParams['axes.labelweight'] = 'bold'
mpl.rcParams['legend.fontsize'] = 14

# ================== STEP 1: NORMALIZE ==================
x = np.array(rng_values, dtype=np.uint32)
u = (x + 0.5) / (2**32)

# ================== STEP 2: VECTORISED BOX-MULLER ==================
# ensure even length from the start
if len(u) % 2 != 0:
    u = u[:-1]

u1 = u[0::2]
u2 = u[1::2]

mask = u1 > 1e-12
u1 = u1[mask]
u2 = u2[mask]

r = np.sqrt(-2 * np.log(u1))
theta = 2 * np.pi * u2

z1 = r * np.cos(theta)
z2 = r * np.sin(theta)

gaussian_numbers = np.concatenate([z1, z2])

# ================== STEP 3: STATISTICS ==================
mu = np.mean(gaussian_numbers)
sigma = np.std(gaussian_numbers)

print(f"Mean = {mu:.4f} (expected ~0)")
print(f"Std Dev = {sigma:.4f} (expected ~1)")

# ================== STEP 4: PLOT ==================
plt.figure(figsize=(10, 6))

# Histogram (normalized)
plt.hist(
    gaussian_numbers,
    bins=80,
    density=True,
    alpha=0.6,
    label='PRNG Gaussian Samples'
)

# Ideal Gaussian curve
x_fit = np.linspace(-5, 5, 1000)
plt.plot(
    x_fit,
    norm.pdf(x_fit, 0, 1),
    'k-',
    linewidth=2,
    label='Ideal N(0,1)'
)

# Your fitted Gaussian
plt.plot(
    x_fit,
    norm.pdf(x_fit, mu, sigma),
    'r--',
    linewidth=2,
    label=f'Fitted N({mu:.2f}, {sigma:.2f})'
)

plt.xlabel('Gaussian Value')
plt.ylabel('Density')
plt.title('Gaussian Distribution from PRNG (Box-Muller Transform)')

plt.legend()
plt.grid(alpha=0.2)
plt.tight_layout()
plt.show()